In [1]:
import torch
from torch import nn
import numpy as np
import json

In [2]:
import os
import sys
sys.path.append('../')
from pqcqec.noise.builder import build_circuit, build_regular_noisy_circuit, create_pqc_circuit_template_simplified, update_pqc_circuit_template, decompile_circuit



In [3]:
DATA_PATH = '../nogit/circuit_tokens/no_uncomp/5q_10g_circuit_data/'
CIRCUIT_DATA_PATH = DATA_PATH + 'per_seed_data/'
CONFIG_PATH = DATA_PATH + 'config.json'

with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

NUM_QUBITS = config['qubits'][0]
NUM_GATES = config['gates'][0]

NUM_GATE_BLOCKS = 5
PQC_BLOCKS = NUM_GATES // NUM_GATE_BLOCKS

circuit_data = []
for filename in os.listdir(CIRCUIT_DATA_PATH):
    with open(CIRCUIT_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        circuit_data.append(token_dict['base_circuit_tokens'])
        f.close()


In [4]:
ideal_circuit_ops_numba_list = []
for tokens in circuit_data:
    ideal_circuit_ops = build_circuit(tokens)
    ideal_circuit_ops_numba_list.append(tuple(ideal_circuit_ops))

print(f"Total Circuits Loaded: {len(ideal_circuit_ops_numba_list)}")
numba_circuit = ideal_circuit_ops_numba_list[0]
print(numba_circuit)

Total Circuits Loaded: 1001
(array([7, 2, 2, 7, 8, 3, 2, 1, 7, 3], dtype=int32), array([4, 0, 0, 0, 1, 0, 3, 4, 4, 4], dtype=int32), array([ 0, -1, -1,  4,  4, -1, -1, -1,  2, -1], dtype=int32), array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32))


In [5]:
pqc_template_numba_circuits = []
x_noise = np.ones((len(circuit_data),NUM_GATES)) * 0.01
z_noise = np.ones((len(circuit_data),NUM_GATES)) * 0.01

for i, tokens in enumerate(circuit_data):
    tagged_noisy_circ = build_regular_noisy_circuit(tokens, x_noise[i], z_noise[i], return_tagged=True)
    pqc_template_dict = create_pqc_circuit_template_simplified(
        tagged_noisy_circ, num_qubits=NUM_QUBITS, gate_blocks=NUM_GATE_BLOCKS, 
        pqc_gates=['rz', 'rx', 'rz'], num_pqc_blocks=PQC_BLOCKS, dtype=np.float32, ignore_noise_gates=True)
    pqc_template_numba_circuits.append(pqc_template_dict)

template_example = pqc_template_numba_circuits[0]
print("PQC Circuit Template:")
print(template_example)

PQC Circuit Template:
{'gate_ids': array([7, 4, 6, 4, 6, 2, 4, 6, 2, 4, 6, 7, 4, 6, 4, 6, 8, 4, 6, 4, 6, 6,
       4, 6, 6, 4, 6, 6, 4, 6, 6, 4, 6, 6, 4, 6, 3, 4, 6, 2, 4, 6, 1, 4,
       6, 7, 4, 6, 4, 6, 3, 4, 6, 6, 4, 6, 6, 4, 6, 6, 4, 6, 6, 4, 6, 6,
       4, 6], dtype=int32), 'wire1': array([4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 1, 1, 1, 4, 4, 0,
       0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 0, 0, 0, 3, 3, 3, 4, 4,
       4, 4, 4, 4, 2, 2, 4, 4, 4, 0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4,
       4, 4], dtype=int32), 'wire2': array([ 0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  4, -1, -1, -1, -1,  4,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  2, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
      dtype=int32), 'theta': array([0.  , 0.01, 0.01, 0.01, 0.01, 0.  , 0.01, 0.01, 0.  , 0.01, 0.01,
       0.  , 0.01, 0.01, 0.01, 0.01, 0.  , 0.01, 0

In [6]:
from pqcqec.training.tokenizer import SimpleCircuitTokenizer
from pqcqec.utils.constants import GATE_IS_DIRECTIONAL, QUBITS_FOR_GATES, GATE_DICT

gate_set = list(GATE_DICT.keys())
undirected_gates = [gate for gate in gate_set if not GATE_IS_DIRECTIONAL.get(gate, False)]
qc_tokenizer = SimpleCircuitTokenizer(
    gateset=gate_set, num_qubits=NUM_QUBITS, qubits_for_gates=QUBITS_FOR_GATES, undirected_gates=undirected_gates
)